In [ ]:
"""
FULLY SELF-CONTAINED CELL -- paste this ENTIRE content into ONE new Jupyter cell and run it.
No other cell needs to run first. This includes every function it needs, inline.

WHAT IT DOES: reprocesses ONLY TARGET_SUBJECT (default "VP004") with stricter bad-segment
rejection before ICA, using the EXACT SAME notch/bandpass/bad-channel/reference/spatial-
mapping/windowing logic as your validated preprocessing_v2_fixed.py (single-band, 1-40Hz
bandpass, single spatial_mapping() call -- NOT the older 5-band decomposition variant), and
overwrites ONLY that subject's output files. It never touches any other subject's data or files.

CHANGED (aligned with preprocessing_v2_fixed.py, see chat history): this version was previously
written against an older 5-band (delta/theta/alpha/beta/gamma) preprocessing variant that used a
0.5-40Hz bandpass and spatial_mapping_multiband() (EEG windows shaped (16,16,5,600)). That no
longer matches your validated single-band pipeline (1.0-40Hz bandpass, plain spatial_mapping(),
EEG windows shaped (16,16,600)). All band-decomposition code has been removed and every filter/
window/shape parameter below now matches preprocessing_v2_fixed.py exactly, so the .npy files
this script overwrites stay fully compatible with the rest of your pipeline (MMapHybridDataset's
memmap shape (N,16,16,600), etc.).

WHAT'S DIFFERENT FROM THE NORMAL PIPELINE: before fitting ICA, segments where any channel
exceeds 100 uV peak-to-peak (stricter than the normal 120 uV check) are annotated as
"BAD_amplitude" and excluded from the ICA fit (reject_by_annotation=True), so large motion
artifacts don't distort the ICA decomposition. This is the ONLY intentional difference from
preprocess_eeg() in preprocessing_v2_fixed.py.

After running, verify with:  run_qc_batch(subjects=[TARGET_SUBJECT], out=r"...")
"""

import os
import numpy as np
import scipy.io as sio
from scipy.signal import resample
from scipy.interpolate import CloughTocher2DInterpolator
from scipy.spatial import Delaunay
import mne
import mne_icalabel

# ============================================================================
# EDIT THESE TO MATCH YOUR MAIN PREPROCESSING SCRIPT EXACTLY
# ============================================================================
TARGET_SUBJECT = "VP004"
BASE_DATASET_PATH = r"E:\kntu\Thesis\codes\DATA"
OUTPUT_PATH       = r"F:\preprocessing-history-thesis\test0.0"
STRICT_AMPLITUDE_UV = 100.0   # tighter than the pipeline's normal 120uV gross-artifact threshold
BAD_SEGMENT_WINDOW_S = 1.0
BAD_SEGMENT_PADDING_S = 0.25
# ============================================================================

# CHANGED: every constant below now matches preprocessing_v2_fixed.py exactly (values, not just names)
SFREQ_EEG = 200.0
SFREQ_FNIRS = 10.0
EEG_WINDOW_LEN = 3.0
EEG_STEP = 1.0
EEG_EPOCH_TMAX = 40.0
FNIRS_WINDOW_LEN = 3.0
FNIRS_NUM_SEGMENTS = 11
FNIRS_STEP = 1.0
BASELINE_TMIN = -5.0
BASELINE_TMAX = -2.0
EEG_TRIAL_LABEL_MAP = {112: 0, 128: 1, 144: 2}
FNIRS_TRIAL_LABEL_MAP = {7: 0, 8: 1, 9: 2}
# REMOVED: EEG_FREQ_BANDS / EEG_BAND_ORDER -- the validated pipeline (preprocessing_v2_fixed.py)
# is SINGLE-BAND (one 1-40Hz bandpass, one spatial_mapping() call producing (16,16,600) windows),
# not the older 5-band delta/theta/alpha/beta/gamma decomposition this script previously used.


# ----------------------------------------------------------------------------
# LOAD FUNCTIONS (copied from your validated preprocessing_v2_fixed.py, unchanged)
# ----------------------------------------------------------------------------

def load_eeg_data(eeg_folder):
    cnt_path = os.path.join(eeg_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(eeg_folder, 'mrk_nback.mat')
    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    cnt_struct = mat_cnt[[k for k in mat_cnt.keys() if not k.startswith('__')][0]]
    data = cnt_struct.x.T
    sfreq = float(cnt_struct.fs)
    ch_names = [str(ch).strip() for ch in cnt_struct.clab]
    if np.max(np.abs(data)) > 1.0:
        data = data * 1e-6
    if sfreq != SFREQ_EEG:
        n_samples = int(data.shape[1] * SFREQ_EEG / sfreq)
        data = resample(data, n_samples, axis=1)
        sfreq = SFREQ_EEG
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(data, info, verbose=False)
    eog_channels = [ch for ch in ['VEOG', 'HEOG'] if ch in ch_names]
    if eog_channels:
        raw.set_channel_types({ch: 'eog' for ch in eog_channels})
    raw.set_montage('standard_1005', match_case=False, on_missing='ignore')

    mat_mrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)
    mrk_struct = mat_mrk[[k for k in mat_mrk.keys() if not k.startswith('__')][0]]
    if hasattr(mrk_struct, 'time'):
        event_positions = np.round(mrk_struct.time * sfreq / 1000).astype(int)
    elif hasattr(mrk_struct, 'pos'):
        event_positions = np.array(mrk_struct.pos, dtype=int) - 1
    else:
        raise ValueError("Could not find 'time' or 'pos' in marker struct")
    if hasattr(mrk_struct, 'event') and hasattr(mrk_struct.event, 'desc'):
        desc = mrk_struct.event.desc
        if desc.ndim > 1:
            desc = desc.squeeze()
        event_codes = np.asarray(desc, dtype=int).flatten()
    else:
        event_codes = event_positions.copy()
    events = np.zeros((len(event_positions), 3), dtype=int)
    events[:, 0] = event_positions
    events[:, 2] = event_codes

    mnt_path = os.path.join(eeg_folder, 'mnt_nback.mat')
    mat_eeg_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)
    eeg_mnt_struct = mat_eeg_mnt[[k for k in mat_eeg_mnt.keys() if not k.startswith('__')][0]]
    return raw, events, sfreq, eeg_mnt_struct


def spatial_mapping(data, mnt):
    n_samples, n_channels = data.shape
    pos_3d = np.asarray(mnt.pos_3d, dtype=np.float64)
    x3d, y3d, z3d = pos_3d[0, :], pos_3d[1, :], pos_3d[2, :]
    r = np.sqrt(x3d**2 + y3d**2 + z3d**2)
    r_safe = np.where(r > 0, r, 1.0)
    theta = np.arccos(np.clip(z3d / r_safe, -1.0, 1.0))
    phi = np.arctan2(y3d, x3d)
    x_coords = theta * np.cos(phi)
    y_coords = theta * np.sin(phi)
    grid_x, grid_y = np.mgrid[min(x_coords):max(x_coords):16j, min(y_coords):max(y_coords):16j]
    points = np.column_stack((x_coords, y_coords))
    tri = Delaunay(points)
    values = data.T
    interp = CloughTocher2DInterpolator(tri, values, fill_value=0)
    result = interp((grid_x, grid_y))
    return np.transpose(result, (2, 0, 1)).astype(np.float32)
# REMOVED: spatial_mapping_multiband() -- not used by the single-band pipeline; the single
# spatial_mapping() function above (identical to preprocessing_v2_fixed.py) is used instead,
# applied once to the full continuous cleaned signal (see reprocessing section at the bottom).


class _EEGMontageAdapter:
    def __init__(self, pos_3d):
        self.pos_3d = pos_3d


def get_eeg_mnt(raw, eeg_mnt_struct):
    full_clab = [str(c).strip() for c in np.asarray(eeg_mnt_struct.clab).flatten()]
    full_pos_3d = np.asarray(eeg_mnt_struct.pos_3d, dtype=np.float64)
    name_to_idx = {name: i for i, name in enumerate(full_clab)}
    name_to_idx_upper = {name.upper(): i for i, name in enumerate(full_clab)}
    pos_list = []
    for ch in raw.ch_names:
        idx = name_to_idx.get(ch, name_to_idx_upper.get(ch.upper(), None))
        pos_list.append([0.0, 0.0, 0.0] if idx is None else full_pos_3d[:, idx])
    return _EEGMontageAdapter(np.array(pos_list, dtype=np.float64).T)


def load_and_preprocess_fnirs(subject_id):
    fnirs_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-NIRS")
    cnt_path = os.path.join(fnirs_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(fnirs_folder, 'mrk_nback.mat')
    mnt_path = os.path.join(fnirs_folder, 'mnt_nback.mat')
    if not os.path.exists(cnt_path):
        raise FileNotFoundError(f"fNIRS file not found: {cnt_path}")

    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    cnt = mat_cnt[[k for k in mat_cnt.keys() if not k.startswith('__')][0]]
    mat_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)
    mnt = mat_mnt[[k for k in mat_mnt.keys() if not k.startswith('__')][0]]

    fnirs_events_time, fnirs_events_code = None, None
    if os.path.exists(mrk_path):
        mat_fmrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)
        fmrk = mat_fmrk[[k for k in mat_fmrk.keys() if not k.startswith('__')][0]]
        if hasattr(fmrk, 'time') and hasattr(fmrk, 'event') and hasattr(fmrk.event, 'desc'):
            fnirs_events_time = np.asarray(fmrk.time).flatten()
            fnirs_events_code = np.asarray(fmrk.event.desc).flatten()

    sfreq_fnirs_raw = float(cnt.oxy.fs) if hasattr(cnt, 'oxy') and hasattr(cnt.oxy, 'fs') else 10.0
    hbo_raw = np.asarray(cnt.oxy.x, dtype=np.float64)
    deoxy_raw = np.asarray(cnt.deoxy.x, dtype=np.float64)
    hbo_filtered, hbr_filtered = hbo_raw, deoxy_raw  # Beer-Lambert already applied; no extra filtering (MSTFDN-style)

    n_samples = int(hbo_filtered.shape[0] * SFREQ_FNIRS / sfreq_fnirs_raw)
    hbo_resampled = resample(hbo_filtered, n_samples, axis=0)
    hbr_resampled = resample(hbr_filtered, n_samples, axis=0)

    hbo_mapped = spatial_mapping(hbo_resampled, mnt)
    hbr_mapped = spatial_mapping(hbr_resampled, mnt)
    fnirs_data = np.stack([hbo_mapped, hbr_mapped], axis=-1)

    return fnirs_data, fnirs_events_time, fnirs_events_code
# REMOVED: apply_band_decomposition() -- not used; the single-band pipeline filters once
# (inside preprocess_eeg_strict below) instead of producing 5 separate band-filtered copies.


def extract_synchronized_windows(eeg_raw, eeg_spatial_data, fnirs_data, events, sfreq_eeg,
                                  fnirs_events_time, fnirs_events_code):
    label_map = EEG_TRIAL_LABEL_MAP
    valid_codes = list(EEG_TRIAL_LABEL_MAP.keys())
    target_events = events[np.isin(events[:, 2], valid_codes)]
    if len(target_events) == 0:
        return np.array([]), np.array([]), np.array([]), np.array([]), np.array([])

    fnirs_sync_ok, fnirs_trial_times = False, None
    if fnirs_events_time is not None and fnirs_events_code is not None:
        fnirs_valid_codes = list(FNIRS_TRIAL_LABEL_MAP.keys())
        fnirs_mask = np.isin(fnirs_events_code, fnirs_valid_codes)
        fnirs_trial_times_ms = np.asarray(fnirs_events_time)[fnirs_mask].astype(float)
        fnirs_trial_codes = np.asarray(fnirs_events_code)[fnirs_mask]
        fnirs_order = np.argsort(fnirs_trial_times_ms)
        fnirs_trial_times = fnirs_trial_times_ms[fnirs_order] / 1000.0
        fnirs_trial_classes = np.array([FNIRS_TRIAL_LABEL_MAP[c] for c in fnirs_trial_codes[fnirs_order]])
        eeg_trial_classes = np.array([label_map.get(int(ev[2]), -1) for ev in target_events])
        n_common = min(len(fnirs_trial_classes), len(eeg_trial_classes))
        if (len(fnirs_trial_times) == len(target_events)
                and np.array_equal(fnirs_trial_classes[:n_common], eeg_trial_classes[:n_common])):
            fnirs_sync_ok = True

    window_samples_eeg = int(EEG_WINDOW_LEN * sfreq_eeg)
    window_samples_fnirs = int(FNIRS_WINDOW_LEN * SFREQ_FNIRS)
    max_slide_seconds = EEG_EPOCH_TMAX - EEG_WINDOW_LEN
    step_seconds = EEG_STEP

    eeg_windows, fnirs_windows, labels, session_ids, trial_ids = [], [], [], [], []
    for trial_idx, ev in enumerate(target_events):
        event_sample_eeg, raw_code = ev[0], ev[2]
        label = label_map.get(raw_code, -1)
        if label == -1:
            continue
        session_id = trial_idx // 9 + 1
        eeg_event_time = event_sample_eeg / sfreq_eeg
        fnirs_event_time = fnirs_trial_times[trial_idx] if fnirs_sync_ok else eeg_event_time

        baseline_start_fnirs = int(np.round((fnirs_event_time + BASELINE_TMIN) * SFREQ_FNIRS))
        baseline_end_fnirs = int(np.round((fnirs_event_time + BASELINE_TMAX) * SFREQ_FNIRS))
        if baseline_start_fnirs >= 0 and baseline_end_fnirs > baseline_start_fnirs:
            fnirs_baseline = fnirs_data[baseline_start_fnirs:baseline_end_fnirs].mean(axis=0, keepdims=True)
        else:
            fnirs_baseline = 0.0

        for offset_sec in np.arange(0.0, max_slide_seconds + 0.001, step_seconds):
            w_start_eeg = int(event_sample_eeg + np.round(offset_sec * sfreq_eeg))
            w_end_eeg = w_start_eeg + window_samples_eeg
            if w_end_eeg > eeg_raw.n_times:
                continue

            fnirs_segments, valid_segments = [], True
            for seg_idx in range(FNIRS_NUM_SEGMENTS):
                seg_offset = offset_sec + seg_idx * FNIRS_STEP
                w_start_fnirs_seg = int(np.round((fnirs_event_time + seg_offset) * SFREQ_FNIRS))
                w_end_fnirs_seg = w_start_fnirs_seg + window_samples_fnirs
                if w_end_fnirs_seg > fnirs_data.shape[0] or w_start_fnirs_seg < 0:
                    valid_segments = False
                    break
                fnirs_segments.append(fnirs_data[w_start_fnirs_seg:w_end_fnirs_seg] - fnirs_baseline)
            if not valid_segments:
                continue

            eeg_slice_raw = eeg_raw.get_data(start=w_start_eeg, stop=w_end_eeg)
            if np.max(np.abs(eeg_slice_raw)) > 120e-6:
                continue

            # CHANGED (single-band alignment): eeg_spatial_data is now (n_times, 16, 16) -- the
            # plain single-band spatial_mapping() output, exactly as in preprocessing_v2_fixed.py --
            # not the old (n_times, 16, 16, 5) multi-band tensor. The slice is therefore
            # transposed (1, 2, 0) -> (16, 16, 600), matching preprocessing_v2_fixed.py exactly,
            # not the old (1, 2, 3, 0) -> (16, 16, 5, 600) multi-band transpose.
            eeg_slice_spatial = eeg_spatial_data[w_start_eeg:w_end_eeg]  # (600, 16, 16)
            eeg_slice = eeg_slice_spatial.transpose(1, 2, 0)             # CHANGED: (16, 16, 600), was transpose(1,2,3,0)
            fnirs_slice = np.stack(fnirs_segments, axis=0).transpose(0, 2, 3, 1, 4)

            eeg_windows.append(eeg_slice)
            fnirs_windows.append(fnirs_slice)
            labels.append(label)
            session_ids.append(session_id)
            trial_ids.append(trial_idx)

    return (np.array(eeg_windows), np.array(fnirs_windows), np.array(labels),
            np.array(session_ids, dtype=np.int32), np.array(trial_ids, dtype=np.int32))


# ----------------------------------------------------------------------------
# STRICT preprocessing (the actual change vs. your normal pipeline)
# ----------------------------------------------------------------------------

def annotate_bad_segments(raw, threshold_uv=STRICT_AMPLITUDE_UV,
                           window_s=BAD_SEGMENT_WINDOW_S, padding_s=BAD_SEGMENT_PADDING_S):
    data = raw.get_data(picks="eeg")
    if np.median(np.abs(data)) < 1e-3:
        data = data * 1e6
    sfreq = raw.info["sfreq"]
    win = int(window_s * sfreq)
    n_windows = data.shape[1] // win
    bad_mask = np.zeros(n_windows, dtype=bool)
    for w in range(n_windows):
        seg = data[:, w * win:(w + 1) * win]
        if np.ptp(seg, axis=1).max() > threshold_uv:
            bad_mask[w] = True
    onsets, durations, i = [], [], 0
    while i < n_windows:
        if bad_mask[i]:
            start = i
            while i < n_windows and bad_mask[i]:
                i += 1
            onset_s = max(0.0, start * window_s - padding_s)
            end_s = min(raw.times[-1], i * window_s + padding_s)
            onsets.append(onset_s)
            durations.append(end_s - onset_s)
        else:
            i += 1
    print(f"  Flagged {len(onsets)} bad segment(s), {sum(durations):.1f}s total out of "
          f"{raw.times[-1]:.1f}s ({sum(durations)/raw.times[-1]*100:.1f}%)")
    if onsets:
        bad_annot = mne.Annotations(onset=onsets, duration=durations,
                                     description=["BAD_amplitude"] * len(onsets))
        raw.set_annotations(raw.annotations + bad_annot if raw.annotations else bad_annot)
    return raw


def preprocess_eeg_strict(raw, events):
    print("Preprocessing EEG (STRICT bad-segment rejection before ICA)...")
    raw_filtered = raw.copy().notch_filter(freqs=50.0, picks="eeg", verbose=False)
    # CHANGED: l_freq 0.5 -> 1.0, matching preprocess_eeg() in preprocessing_v2_fixed.py exactly
    # (this project's validated highpass cutoff is 1.0Hz, not 0.5Hz).
    raw_filtered.filter(l_freq=1.0, h_freq=40.0, method="fir", verbose=False)  # CHANGED: was l_freq=0.5

    raw_temp = raw_filtered.copy().filter(l_freq=10.0, h_freq=40.0, method="fir", verbose=False)
    eeg_picks = mne.pick_types(raw_temp.info, eeg=True, eog=False)
    eeg_names = [raw_temp.ch_names[idx] for idx in eeg_picks]
    temp_data = raw_temp.get_data(picks=eeg_picks)
    stds = np.std(temp_data, axis=1)
    median_std = np.median(stds)
    mad = np.median(np.abs(stds - median_std)) if np.median(np.abs(stds - median_std)) > 0 else 1e-6
    corr_matrix = np.corrcoef(temp_data)
    mean_corrs = np.mean(corr_matrix, axis=1)
    bad_channels = []
    for i, ch_name in enumerate(eeg_names):
        if ch_name.upper() in ["VEOG", "HEOG", "TP9"]:
            continue
        if stds[i] > (median_std + 4.5 * mad) or stds[i] < (median_std - 4.5 * mad) or mean_corrs[i] < 0.30:
            bad_channels.append(ch_name)
    if bad_channels:
        print(f"Bad channels: {bad_channels}")
        raw_filtered.info["bads"] = bad_channels
    if raw_filtered.info["bads"]:
        raw_filtered.interpolate_bads(reset_bads=True, verbose=False)
    raw_filtered.set_eeg_reference("average", projection=False, verbose=False)

    print("  Annotating high-amplitude segments for exclusion from ICA fit...")
    raw_filtered = annotate_bad_segments(raw_filtered)

    picks_for_ica = mne.pick_types(raw_filtered.info, eeg=True, eog=False, exclude="bads")
    ica = mne.preprocessing.ICA(n_components=0.99, method="picard",
                                 fit_params=dict(ortho=False, extended=True),
                                 random_state=42, verbose=False)
    ica.fit(raw_filtered, picks=picks_for_ica, reject_by_annotation=True)  # THE ACTUAL CHANGE: strict bad-segment exclusion from ICA fit

    ic_labels = mne_icalabel.label_components(raw_filtered, ica, method="iclabel")
    ARTIFACT_CLASSES_IDX = {"brain": 0, "muscle artifact": 1, "eye blink": 2,
                             "heart beat": 3, "line noise": 4, "channel noise": 5, "other": 6}
    ARTIFACT_TARGETS = ["eye blink", "eye movement", "muscle artifact", "line noise"]
    PROB_THRESHOLD = 0.30
    artifact_idx_argmax = [idx for idx, label in enumerate(ic_labels["labels"]) if label in ARTIFACT_TARGETS]
    artifact_idx_prob = []
    proba_matrix = ica.labels_scores_
    for comp_idx in range(proba_matrix.shape[0]):
        for cls_name in ARTIFACT_TARGETS:
            cls_col = ARTIFACT_CLASSES_IDX.get(cls_name)
            if cls_col is not None and proba_matrix[comp_idx, cls_col] >= PROB_THRESHOLD:
                artifact_idx_prob.append(comp_idx)
                break
    eog_indices_veog, eog_indices_heog = [], []
    try:
        if "VEOG" in raw.ch_names:
            eog_indices_veog, _ = ica.find_bads_eog(raw, ch_name="VEOG", threshold=2.0, verbose=False)
        if "HEOG" in raw.ch_names:
            eog_indices_heog, _ = ica.find_bads_eog(raw, ch_name="HEOG", threshold=2.0, verbose=False)
    except Exception as e:
        print(f"  find_bads_eog failed: {e}")
    BRAIN_PROB_CEILING = 0.5
    BRAIN_CLASS_COL = ARTIFACT_CLASSES_IDX["brain"]
    eog_only_idx = (set(eog_indices_veog) | set(eog_indices_heog)) - (set(artifact_idx_argmax) | set(artifact_idx_prob))
    eog_only_rescued = {idx for idx in eog_only_idx if proba_matrix[idx, BRAIN_CLASS_COL] >= BRAIN_PROB_CEILING}
    artifact_idx = sorted(
        (set(artifact_idx_argmax) | set(artifact_idx_prob) | set(eog_indices_veog) | set(eog_indices_heog))
        - eog_only_rescued
    )
    ica.exclude = artifact_idx
    if len(ica.exclude) > 0:
        ica.apply(raw_filtered, verbose=False)
        print(f"Removed {len(ica.exclude)} ICA components: {artifact_idx}")
    else:
        print("Removed 0 ICA components")

    channels_to_drop = [ch for ch in raw_filtered.ch_names if ch.upper() in ["VEOG", "HEOG", "TP9"]]
    if channels_to_drop:
        raw_filtered.drop_channels(channels_to_drop)
        print(f"Dropped channels: {channels_to_drop}")

    return raw_filtered


# ----------------------------------------------------------------------------
# MAIN: reprocess only TARGET_SUBJECT
# ----------------------------------------------------------------------------
print(f"\n{'='*80}\nSTRICT reprocessing: {TARGET_SUBJECT} (does not affect any other subject)\n{'='*80}")

eeg_folder = os.path.join(BASE_DATASET_PATH, f"{TARGET_SUBJECT}-EEG")
raw_eeg, events, sfreq_eeg, eeg_mnt_struct = load_eeg_data(eeg_folder)
fnirs_data, fnirs_events_time, fnirs_events_code = load_and_preprocess_fnirs(TARGET_SUBJECT)
fnirs_data = (fnirs_data - fnirs_data.mean(axis=0, keepdims=True)) / (fnirs_data.std(axis=0, keepdims=True) + 1e-8)

raw_eeg_clean = preprocess_eeg_strict(raw_eeg, events)

sub_out_dir = os.path.join(OUTPUT_PATH, TARGET_SUBJECT)
os.makedirs(sub_out_dir, exist_ok=True)
raw_eeg_clean.save(os.path.join(sub_out_dir, f"{TARGET_SUBJECT}_eeg_clean_raw.fif"), overwrite=True)
print(f"Saved strict-cleaned continuous EEG to {sub_out_dir}\\{TARGET_SUBJECT}_eeg_clean_raw.fif")

# CHANGED (single-band alignment): the old 5-band decomposition + spatial_mapping_multiband()
# block is replaced with the SAME single-band spatial mapping used in preprocessing_v2_fixed.py --
# one spatial_mapping() call on the full continuous, already-cleaned (1-40Hz, ICA'd) signal,
# followed by the same per-subject z-score normalization (mean/std over the time axis, axis=0).
eeg_mnt = get_eeg_mnt(raw_eeg_clean, eeg_mnt_struct)                       # CHANGED: no band_signals/band_data_list needed
eeg_full_data = raw_eeg_clean.get_data()                                    # (n_channels, n_times)
eeg_spatial_full = spatial_mapping(eeg_full_data.T, eeg_mnt)                # CHANGED: single spatial_mapping() call, matches preprocessing_v2_fixed.py exactly -> (n_times, 16, 16)
eeg_spatial_full = (eeg_spatial_full - eeg_spatial_full.mean(axis=0, keepdims=True)) / (
    eeg_spatial_full.std(axis=0, keepdims=True) + 1e-8)                     # per-subject z-score, same as preprocessing_v2_fixed.py

eeg_windows, fnirs_windows, labels, session_ids, trial_ids = extract_synchronized_windows(
    raw_eeg_clean, eeg_spatial_full, fnirs_data, events, sfreq_eeg, fnirs_events_time, fnirs_events_code
)

if len(labels) == 0:
    print(f"No windows extracted for {TARGET_SUBJECT} -- ABORTING save (old files left untouched).")
else:
    np.save(os.path.join(sub_out_dir, f"{TARGET_SUBJECT}_eeg_windows.npy"), eeg_windows)
    np.save(os.path.join(sub_out_dir, f"{TARGET_SUBJECT}_fnirs_windows.npy"), fnirs_windows)
    np.save(os.path.join(sub_out_dir, f"{TARGET_SUBJECT}_labels.npy"), labels)
    np.save(os.path.join(sub_out_dir, f"{TARGET_SUBJECT}_session_ids.npy"), session_ids)
    np.save(os.path.join(sub_out_dir, f"{TARGET_SUBJECT}_trial_ids.npy"), trial_ids)

    print(f"\n{TARGET_SUBJECT} strict-reprocessing complete!")
    print(f"EEG windows: {eeg_windows.shape}")  # CHANGED: now (N, 16, 16, 600), matching preprocessing_v2_fixed.py -- was (N, 16, 16, 5, 600)
    print(f"fNIRS windows: {fnirs_windows.shape}")
    print(f"Labels: {np.bincount(labels)}")
    print(f"\nNext step: re-run QC for {TARGET_SUBJECT} only to confirm the FAIL/WARN items are resolved:")
    print(f'  run_qc_batch(subjects=["{TARGET_SUBJECT}"], out=r"...")')


In [ ]:
"""
FULLY SELF-CONTAINED CELL -- paste this ENTIRE content into ONE new Jupyter cell and run it.

WHAT IT DOES: reprocesses VP015, VP018, VP019, VP020 with Fp1 (and Fp2 where flagged) FORCED as
bad channels -- i.e. spherical-spline interpolated from neighbouring electrodes -- BEFORE ICA,
instead of relying on ICA to remove the residual low-frequency contamination there (which the
ICA eye-blink probability deep-dive confirmed does NOT come from a separable ICA source in any
of these 4 subjects). Everything else is identical to your validated preprocessing_v2_fixed.py.

CHANGED (aligned with preprocessing_v2_fixed.py, see chat history): this version was previously
written against an older 5-band (delta/theta/alpha/beta/gamma) preprocessing variant that used a
0.5-40Hz bandpass and spatial_mapping_multiband() (EEG windows shaped (16,16,5,600)). That no
longer matches your validated single-band pipeline (1.0-40Hz bandpass, plain spatial_mapping(),
EEG windows shaped (16,16,600)). All band-decomposition code has been removed and every filter/
window/shape parameter below now matches preprocessing_v2_fixed.py exactly, so the .npy files
this script overwrites stay fully compatible with the rest of your pipeline (MMapHybridDataset's
memmap shape (N,16,16,600), etc.).

Per-subject forced-bad-channel list (from the delta-band cross-check results):
    VP015: Fp1, Fp2
    VP018: Fp1, Fp2
    VP019: Fp1          (Fp2 was NOT flagged for VP019)
    VP020: Fp1, Fp2

This ONLY overwrites these 4 subjects' output files in OUTPUT_PATH. No other subject is touched.

After running, verify with:
    run_qc_batch(subjects=["VP015","VP018","VP019","VP020"], out=r"...")
and/or re-run delta_band_ocular_crosscheck.py restricted to these 4 subjects to confirm the
elevated Fp1/Fp2 delta power is gone.
"""

import os
import numpy as np
import scipy.io as sio
from scipy.signal import resample
from scipy.interpolate import CloughTocher2DInterpolator
from scipy.spatial import Delaunay
import mne
import mne_icalabel

# ============================================================================
# EDIT THESE TO MATCH YOUR MAIN PREPROCESSING SCRIPT EXACTLY
# ============================================================================
BASE_DATASET_PATH = r"E:\kntu\Thesis\codes\DATA"
OUTPUT_PATH       = r"F:\preprocessing-history-thesis\test0.0"

FORCED_BAD_CHANNELS = {
    "VP015": ["Fp1", "Fp2"],
    "VP018": ["Fp1", "Fp2"],
    "VP019": ["Fp1"],
    "VP020": ["Fp1", "Fp2"],
}
# ============================================================================

# CHANGED: every constant below now matches preprocessing_v2_fixed.py exactly (values, not just names)
SFREQ_EEG = 200.0
SFREQ_FNIRS = 10.0
EEG_WINDOW_LEN = 3.0
EEG_STEP = 1.0
EEG_EPOCH_TMAX = 40.0
FNIRS_WINDOW_LEN = 3.0
FNIRS_NUM_SEGMENTS = 11
FNIRS_STEP = 1.0
BASELINE_TMIN = -5.0
BASELINE_TMAX = -2.0
EEG_TRIAL_LABEL_MAP = {112: 0, 128: 1, 144: 2}
FNIRS_TRIAL_LABEL_MAP = {7: 0, 8: 1, 9: 2}
# REMOVED: EEG_FREQ_BANDS / EEG_BAND_ORDER -- the validated pipeline (preprocessing_v2_fixed.py)
# is SINGLE-BAND (one 1-40Hz bandpass, one spatial_mapping() call producing (16,16,600) windows),
# not the older 5-band delta/theta/alpha/beta/gamma decomposition this script previously used.


# ----------------------------------------------------------------------------
# LOAD / SPATIAL FUNCTIONS (identical to your validated preprocessing_v2_fixed.py)
# ----------------------------------------------------------------------------

def load_eeg_data(eeg_folder):
    cnt_path = os.path.join(eeg_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(eeg_folder, 'mrk_nback.mat')
    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    cnt_struct = mat_cnt[[k for k in mat_cnt.keys() if not k.startswith('__')][0]]
    data = cnt_struct.x.T
    sfreq = float(cnt_struct.fs)
    ch_names = [str(ch).strip() for ch in cnt_struct.clab]
    if np.max(np.abs(data)) > 1.0:
        data = data * 1e-6
    if sfreq != SFREQ_EEG:
        n_samples = int(data.shape[1] * SFREQ_EEG / sfreq)
        data = resample(data, n_samples, axis=1)
        sfreq = SFREQ_EEG
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(data, info, verbose=False)
    eog_channels = [ch for ch in ['VEOG', 'HEOG'] if ch in ch_names]
    if eog_channels:
        raw.set_channel_types({ch: 'eog' for ch in eog_channels})
    raw.set_montage('standard_1005', match_case=False, on_missing='ignore')

    mat_mrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)
    mrk_struct = mat_mrk[[k for k in mat_mrk.keys() if not k.startswith('__')][0]]
    if hasattr(mrk_struct, 'time'):
        event_positions = np.round(mrk_struct.time * sfreq / 1000).astype(int)
    elif hasattr(mrk_struct, 'pos'):
        event_positions = np.array(mrk_struct.pos, dtype=int) - 1
    else:
        raise ValueError("Could not find 'time' or 'pos' in marker struct")
    if hasattr(mrk_struct, 'event') and hasattr(mrk_struct.event, 'desc'):
        desc = mrk_struct.event.desc
        if desc.ndim > 1:
            desc = desc.squeeze()
        event_codes = np.asarray(desc, dtype=int).flatten()
    else:
        event_codes = event_positions.copy()
    events = np.zeros((len(event_positions), 3), dtype=int)
    events[:, 0] = event_positions
    events[:, 2] = event_codes

    mnt_path = os.path.join(eeg_folder, 'mnt_nback.mat')
    mat_eeg_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)
    eeg_mnt_struct = mat_eeg_mnt[[k for k in mat_eeg_mnt.keys() if not k.startswith('__')][0]]
    return raw, events, sfreq, eeg_mnt_struct


def spatial_mapping(data, mnt):
    n_samples, n_channels = data.shape
    pos_3d = np.asarray(mnt.pos_3d, dtype=np.float64)
    x3d, y3d, z3d = pos_3d[0, :], pos_3d[1, :], pos_3d[2, :]
    r = np.sqrt(x3d**2 + y3d**2 + z3d**2)
    r_safe = np.where(r > 0, r, 1.0)
    theta = np.arccos(np.clip(z3d / r_safe, -1.0, 1.0))
    phi = np.arctan2(y3d, x3d)
    x_coords = theta * np.cos(phi)
    y_coords = theta * np.sin(phi)
    grid_x, grid_y = np.mgrid[min(x_coords):max(x_coords):16j, min(y_coords):max(y_coords):16j]
    points = np.column_stack((x_coords, y_coords))
    tri = Delaunay(points)
    values = data.T
    interp = CloughTocher2DInterpolator(tri, values, fill_value=0)
    result = interp((grid_x, grid_y))
    return np.transpose(result, (2, 0, 1)).astype(np.float32)
# REMOVED: spatial_mapping_multiband() -- not used by the single-band pipeline; the single
# spatial_mapping() function above (identical to preprocessing_v2_fixed.py) is used instead,
# applied once to the full continuous cleaned signal (see reprocess_subject_with_forced_bad below).


class _EEGMontageAdapter:
    def __init__(self, pos_3d):
        self.pos_3d = pos_3d


def get_eeg_mnt(raw, eeg_mnt_struct):
    full_clab = [str(c).strip() for c in np.asarray(eeg_mnt_struct.clab).flatten()]
    full_pos_3d = np.asarray(eeg_mnt_struct.pos_3d, dtype=np.float64)
    name_to_idx = {name: i for i, name in enumerate(full_clab)}
    name_to_idx_upper = {name.upper(): i for i, name in enumerate(full_clab)}
    pos_list = []
    for ch in raw.ch_names:
        idx = name_to_idx.get(ch, name_to_idx_upper.get(ch.upper(), None))
        pos_list.append([0.0, 0.0, 0.0] if idx is None else full_pos_3d[:, idx])
    return _EEGMontageAdapter(np.array(pos_list, dtype=np.float64).T)


def load_and_preprocess_fnirs(subject_id):
    fnirs_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-NIRS")
    cnt_path = os.path.join(fnirs_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(fnirs_folder, 'mrk_nback.mat')
    mnt_path = os.path.join(fnirs_folder, 'mnt_nback.mat')
    if not os.path.exists(cnt_path):
        raise FileNotFoundError(f"fNIRS file not found: {cnt_path}")

    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    cnt = mat_cnt[[k for k in mat_cnt.keys() if not k.startswith('__')][0]]
    mat_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)
    mnt = mat_mnt[[k for k in mat_mnt.keys() if not k.startswith('__')][0]]

    fnirs_events_time, fnirs_events_code = None, None
    if os.path.exists(mrk_path):
        mat_fmrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)
        fmrk = mat_fmrk[[k for k in mat_fmrk.keys() if not k.startswith('__')][0]]
        if hasattr(fmrk, 'time') and hasattr(fmrk, 'event') and hasattr(fmrk.event, 'desc'):
            fnirs_events_time = np.asarray(fmrk.time).flatten()
            fnirs_events_code = np.asarray(fmrk.event.desc).flatten()

    sfreq_fnirs_raw = float(cnt.oxy.fs) if hasattr(cnt, 'oxy') and hasattr(cnt.oxy, 'fs') else 10.0
    hbo_raw = np.asarray(cnt.oxy.x, dtype=np.float64)
    deoxy_raw = np.asarray(cnt.deoxy.x, dtype=np.float64)
    hbo_filtered, hbr_filtered = hbo_raw, deoxy_raw  # Beer-Lambert already applied; no extra filtering (MSTFDN-style)

    n_samples = int(hbo_filtered.shape[0] * SFREQ_FNIRS / sfreq_fnirs_raw)
    hbo_resampled = resample(hbo_filtered, n_samples, axis=0)
    hbr_resampled = resample(hbr_filtered, n_samples, axis=0)

    hbo_mapped = spatial_mapping(hbo_resampled, mnt)
    hbr_mapped = spatial_mapping(hbr_resampled, mnt)
    fnirs_data = np.stack([hbo_mapped, hbr_mapped], axis=-1)

    return fnirs_data, fnirs_events_time, fnirs_events_code
# REMOVED: apply_band_decomposition() -- not used; the single-band pipeline filters once
# (inside preprocess_eeg_force_bad below) instead of producing 5 separate band-filtered copies.


def extract_synchronized_windows(eeg_raw, eeg_spatial_data, fnirs_data, events, sfreq_eeg,
                                  fnirs_events_time, fnirs_events_code):
    label_map = EEG_TRIAL_LABEL_MAP
    valid_codes = list(EEG_TRIAL_LABEL_MAP.keys())
    target_events = events[np.isin(events[:, 2], valid_codes)]
    if len(target_events) == 0:
        return np.array([]), np.array([]), np.array([]), np.array([]), np.array([])

    fnirs_sync_ok, fnirs_trial_times = False, None
    if fnirs_events_time is not None and fnirs_events_code is not None:
        fnirs_valid_codes = list(FNIRS_TRIAL_LABEL_MAP.keys())
        fnirs_mask = np.isin(fnirs_events_code, fnirs_valid_codes)
        fnirs_trial_times_ms = np.asarray(fnirs_events_time)[fnirs_mask].astype(float)
        fnirs_trial_codes = np.asarray(fnirs_events_code)[fnirs_mask]
        fnirs_order = np.argsort(fnirs_trial_times_ms)
        fnirs_trial_times = fnirs_trial_times_ms[fnirs_order] / 1000.0
        fnirs_trial_classes = np.array([FNIRS_TRIAL_LABEL_MAP[c] for c in fnirs_trial_codes[fnirs_order]])
        eeg_trial_classes = np.array([label_map.get(int(ev[2]), -1) for ev in target_events])
        n_common = min(len(fnirs_trial_classes), len(eeg_trial_classes))
        if (len(fnirs_trial_times) == len(target_events)
                and np.array_equal(fnirs_trial_classes[:n_common], eeg_trial_classes[:n_common])):
            fnirs_sync_ok = True

    window_samples_eeg = int(EEG_WINDOW_LEN * sfreq_eeg)
    window_samples_fnirs = int(FNIRS_WINDOW_LEN * SFREQ_FNIRS)
    max_slide_seconds = EEG_EPOCH_TMAX - EEG_WINDOW_LEN
    step_seconds = EEG_STEP

    eeg_windows, fnirs_windows, labels, session_ids, trial_ids = [], [], [], [], []
    for trial_idx, ev in enumerate(target_events):
        event_sample_eeg, raw_code = ev[0], ev[2]
        label = label_map.get(raw_code, -1)
        if label == -1:
            continue
        session_id = trial_idx // 9 + 1
        eeg_event_time = event_sample_eeg / sfreq_eeg
        fnirs_event_time = fnirs_trial_times[trial_idx] if fnirs_sync_ok else eeg_event_time

        baseline_start_fnirs = int(np.round((fnirs_event_time + BASELINE_TMIN) * SFREQ_FNIRS))
        baseline_end_fnirs = int(np.round((fnirs_event_time + BASELINE_TMAX) * SFREQ_FNIRS))
        if baseline_start_fnirs >= 0 and baseline_end_fnirs > baseline_start_fnirs:
            fnirs_baseline = fnirs_data[baseline_start_fnirs:baseline_end_fnirs].mean(axis=0, keepdims=True)
        else:
            fnirs_baseline = 0.0

        for offset_sec in np.arange(0.0, max_slide_seconds + 0.001, step_seconds):
            w_start_eeg = int(event_sample_eeg + np.round(offset_sec * sfreq_eeg))
            w_end_eeg = w_start_eeg + window_samples_eeg
            if w_end_eeg > eeg_raw.n_times:
                continue

            fnirs_segments, valid_segments = [], True
            for seg_idx in range(FNIRS_NUM_SEGMENTS):
                seg_offset = offset_sec + seg_idx * FNIRS_STEP
                w_start_fnirs_seg = int(np.round((fnirs_event_time + seg_offset) * SFREQ_FNIRS))
                w_end_fnirs_seg = w_start_fnirs_seg + window_samples_fnirs
                if w_end_fnirs_seg > fnirs_data.shape[0] or w_start_fnirs_seg < 0:
                    valid_segments = False
                    break
                fnirs_segments.append(fnirs_data[w_start_fnirs_seg:w_end_fnirs_seg] - fnirs_baseline)
            if not valid_segments:
                continue

            eeg_slice_raw = eeg_raw.get_data(start=w_start_eeg, stop=w_end_eeg)
            if np.max(np.abs(eeg_slice_raw)) > 120e-6:
                continue

            # CHANGED (single-band alignment): eeg_spatial_data is now (n_times, 16, 16) -- the
            # plain single-band spatial_mapping() output, exactly as in preprocessing_v2_fixed.py --
            # not the old (n_times, 16, 16, 5) multi-band tensor. The slice is therefore
            # transposed (1, 2, 0) -> (16, 16, 600), matching preprocessing_v2_fixed.py exactly,
            # not the old (1, 2, 3, 0) -> (16, 16, 5, 600) multi-band transpose.
            eeg_slice_spatial = eeg_spatial_data[w_start_eeg:w_end_eeg]  # (600, 16, 16)
            eeg_slice = eeg_slice_spatial.transpose(1, 2, 0)             # CHANGED: (16, 16, 600), was transpose(1,2,3,0)
            fnirs_slice = np.stack(fnirs_segments, axis=0).transpose(0, 2, 3, 1, 4)

            eeg_windows.append(eeg_slice)
            fnirs_windows.append(fnirs_slice)
            labels.append(label)
            session_ids.append(session_id)
            trial_ids.append(trial_idx)

    return (np.array(eeg_windows), np.array(fnirs_windows), np.array(labels),
            np.array(session_ids, dtype=np.int32), np.array(trial_ids, dtype=np.int32))


# ----------------------------------------------------------------------------
# preprocess_eeg WITH FORCED BAD CHANNELS (the actual change vs. your normal pipeline)
# ----------------------------------------------------------------------------

def preprocess_eeg_force_bad(raw, events, forced_bad_channels):
    print(f"Preprocessing EEG (forcing {forced_bad_channels} as bad -> interpolated before ICA)...")
    raw_filtered = raw.copy().notch_filter(freqs=50.0, picks="eeg", verbose=False)
    # CHANGED: l_freq 0.5 -> 1.0, matching preprocess_eeg() in preprocessing_v2_fixed.py exactly
    # (this project's validated highpass cutoff is 1.0Hz, not 0.5Hz).
    raw_filtered.filter(l_freq=1.0, h_freq=40.0, method="fir", verbose=False)  # CHANGED: was l_freq=0.5

    raw_temp = raw_filtered.copy().filter(l_freq=10.0, h_freq=40.0, method="fir", verbose=False)
    eeg_picks = mne.pick_types(raw_temp.info, eeg=True, eog=False)
    eeg_names = [raw_temp.ch_names[idx] for idx in eeg_picks]
    temp_data = raw_temp.get_data(picks=eeg_picks)
    stds = np.std(temp_data, axis=1)
    median_std = np.median(stds)
    mad = np.median(np.abs(stds - median_std)) if np.median(np.abs(stds - median_std)) > 0 else 1e-6
    corr_matrix = np.corrcoef(temp_data)
    mean_corrs = np.mean(corr_matrix, axis=1)
    bad_channels = []
    for i, ch_name in enumerate(eeg_names):
        if ch_name.upper() in ["VEOG", "HEOG", "TP9"]:
            continue
        if stds[i] > (median_std + 4.5 * mad) or stds[i] < (median_std - 4.5 * mad) or mean_corrs[i] < 0.30:
            bad_channels.append(ch_name)

    # THE ACTUAL CHANGE for this variant: force Fp1/Fp2 (diagnosed root cause -- see chat history)
    # into the bad-channel list in ADDITION to whatever the automatic std/correlation detector
    # found, so they get spherical-spline interpolated from neighbouring electrodes exactly like
    # any other bad channel, BEFORE ICA -- instead of relying on ICA to remove their residual
    # low-frequency contamination, which the ICA eye-blink probability deep-dive confirmed does
    # NOT come from any separable ICA source in these subjects.
    for ch in forced_bad_channels:
        if ch in raw_filtered.ch_names and ch not in bad_channels:
            bad_channels.append(ch)

    if bad_channels:
        print(f"Bad channels (auto-detected + forced): {bad_channels}")
        raw_filtered.info["bads"] = bad_channels
    if raw_filtered.info["bads"]:
        raw_filtered.interpolate_bads(reset_bads=True, verbose=False)
    raw_filtered.set_eeg_reference("average", projection=False, verbose=False)

    picks_for_ica = mne.pick_types(raw_filtered.info, eeg=True, eog=False, exclude="bads")
    ica = mne.preprocessing.ICA(n_components=0.99, method="picard",
                                 fit_params=dict(ortho=False, extended=True),
                                 random_state=42, verbose=False)
    ica.fit(raw_filtered, picks=picks_for_ica)

    ic_labels = mne_icalabel.label_components(raw_filtered, ica, method="iclabel")
    ARTIFACT_CLASSES_IDX = {"brain": 0, "muscle artifact": 1, "eye blink": 2,
                             "heart beat": 3, "line noise": 4, "channel noise": 5, "other": 6}
    ARTIFACT_TARGETS = ["eye blink", "eye movement", "muscle artifact", "line noise"]
    PROB_THRESHOLD = 0.30
    artifact_idx_argmax = [idx for idx, label in enumerate(ic_labels["labels"]) if label in ARTIFACT_TARGETS]
    artifact_idx_prob = []
    proba_matrix = ica.labels_scores_
    for comp_idx in range(proba_matrix.shape[0]):
        for cls_name in ARTIFACT_TARGETS:
            cls_col = ARTIFACT_CLASSES_IDX.get(cls_name)
            if cls_col is not None and proba_matrix[comp_idx, cls_col] >= PROB_THRESHOLD:
                artifact_idx_prob.append(comp_idx)
                break
    eog_indices_veog, eog_indices_heog = [], []
    try:
        if "VEOG" in raw.ch_names:
            eog_indices_veog, _ = ica.find_bads_eog(raw, ch_name="VEOG", threshold=2.0, verbose=False)
        if "HEOG" in raw.ch_names:
            eog_indices_heog, _ = ica.find_bads_eog(raw, ch_name="HEOG", threshold=2.0, verbose=False)
    except Exception as e:
        print(f"  find_bads_eog failed: {e}")
    BRAIN_PROB_CEILING = 0.5
    BRAIN_CLASS_COL = ARTIFACT_CLASSES_IDX["brain"]
    eog_only_idx = (set(eog_indices_veog) | set(eog_indices_heog)) - (set(artifact_idx_argmax) | set(artifact_idx_prob))
    eog_only_rescued = {idx for idx in eog_only_idx if proba_matrix[idx, BRAIN_CLASS_COL] >= BRAIN_PROB_CEILING}
    artifact_idx = sorted(
        (set(artifact_idx_argmax) | set(artifact_idx_prob) | set(eog_indices_veog) | set(eog_indices_heog))
        - eog_only_rescued
    )
    ica.exclude = artifact_idx
    if len(ica.exclude) > 0:
        ica.apply(raw_filtered, verbose=False)
        print(f"Removed {len(ica.exclude)} ICA components: {artifact_idx}")
    else:
        print("Removed 0 ICA components")

    channels_to_drop = [ch for ch in raw_filtered.ch_names if ch.upper() in ["VEOG", "HEOG", "TP9"]]
    if channels_to_drop:
        raw_filtered.drop_channels(channels_to_drop)
        print(f"Dropped channels: {channels_to_drop}")

    return raw_filtered


def reprocess_subject_with_forced_bad(subject_id, forced_bad_channels):
    print(f"\n{'='*90}\nReprocessing {subject_id} (forced bad: {forced_bad_channels}) "
          f"-- does not affect any other subject\n{'='*90}")

    eeg_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-EEG")
    raw_eeg, events, sfreq_eeg, eeg_mnt_struct = load_eeg_data(eeg_folder)
    fnirs_data, fnirs_events_time, fnirs_events_code = load_and_preprocess_fnirs(subject_id)
    fnirs_data = (fnirs_data - fnirs_data.mean(axis=0, keepdims=True)) / (fnirs_data.std(axis=0, keepdims=True) + 1e-8)

    raw_eeg_clean = preprocess_eeg_force_bad(raw_eeg, events, forced_bad_channels)

    sub_out_dir = os.path.join(OUTPUT_PATH, subject_id)
    os.makedirs(sub_out_dir, exist_ok=True)
    raw_eeg_clean.save(os.path.join(sub_out_dir, f"{subject_id}_eeg_clean_raw.fif"), overwrite=True)
    print(f"Saved cleaned EEG to {sub_out_dir}\\{subject_id}_eeg_clean_raw.fif")

    # CHANGED (single-band alignment): the old 5-band decomposition + spatial_mapping_multiband()
    # block is replaced with the SAME single-band spatial mapping used in preprocessing_v2_fixed.py --
    # one spatial_mapping() call on the full continuous, already-cleaned (1-40Hz, ICA'd) signal,
    # followed by the same per-subject z-score normalization (mean/std over the time axis, axis=0).
    eeg_mnt = get_eeg_mnt(raw_eeg_clean, eeg_mnt_struct)                       # CHANGED: no band_signals/band_data_list needed
    eeg_full_data = raw_eeg_clean.get_data()                                    # (n_channels, n_times)
    eeg_spatial_full = spatial_mapping(eeg_full_data.T, eeg_mnt)                # CHANGED: single spatial_mapping() call, matches preprocessing_v2_fixed.py exactly -> (n_times, 16, 16)
    eeg_spatial_full = (eeg_spatial_full - eeg_spatial_full.mean(axis=0, keepdims=True)) / (
        eeg_spatial_full.std(axis=0, keepdims=True) + 1e-8)                     # per-subject z-score, same as preprocessing_v2_fixed.py

    eeg_windows, fnirs_windows, labels, session_ids, trial_ids = extract_synchronized_windows(
        raw_eeg_clean, eeg_spatial_full, fnirs_data, events, sfreq_eeg, fnirs_events_time, fnirs_events_code
    )

    if len(labels) == 0:
        print(f"No windows extracted for {subject_id} -- ABORTING save (old files left untouched).")
        return

    np.save(os.path.join(sub_out_dir, f"{subject_id}_eeg_windows.npy"), eeg_windows)
    np.save(os.path.join(sub_out_dir, f"{subject_id}_fnirs_windows.npy"), fnirs_windows)
    np.save(os.path.join(sub_out_dir, f"{subject_id}_labels.npy"), labels)
    np.save(os.path.join(sub_out_dir, f"{subject_id}_session_ids.npy"), session_ids)
    np.save(os.path.join(sub_out_dir, f"{subject_id}_trial_ids.npy"), trial_ids)

    print(f"\n{subject_id} reprocessing complete!")
    print(f"EEG windows: {eeg_windows.shape}")  # CHANGED: now (N, 16, 16, 600), matching preprocessing_v2_fixed.py -- was (N, 16, 16, 5, 600)
    print(f"fNIRS windows: {fnirs_windows.shape}")
    print(f"Labels: {np.bincount(labels)}")


# ----------------------------------------------------------------------------
# RUN for all 4 target subjects
# ----------------------------------------------------------------------------
for subject_id, forced_bad in FORCED_BAD_CHANNELS.items():
    try:
        reprocess_subject_with_forced_bad(subject_id, forced_bad)
    except Exception as e:
        print(f"\n{subject_id}: FAILED -- {e}")
        import traceback
        traceback.print_exc()

print(f"\n\n{'='*90}")
print("All 4 subjects reprocessed. Next step -- verify with:")
print('  run_qc_batch(subjects=["VP015","VP018","VP019","VP020"], out=r"...")')
print(f"{'='*90}")
